💡 **Environment:** `clamp-analyses`  


# Description

Predicts drug-disease associations using the ARCHS4 **module-based** approach: CLAMP-projected S-PrediXcan (disease, 49 tissues) and LINCS L1000 (drug) in the ARCHS4 latent space.

For each of 49 tissues and 5 LV-count thresholds (all, 5, 10, 25, 50), runs:
$$\text{score} = -1 \times \mathbf{drug}^T \mathbf{disease}$$
in the LV space.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [ ]:
PREDICTION_METHOD = 'module_based_archs4'

In [4]:
DATA_DIR = here('data/drug_disease_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

NB_NAME = '07_prediction_module_based_archs4'
OUTPUT_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/' + NB_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inputs from upstream notebooks
LINCS_PROJ_FILE = here('output/03_model_biology/00_archs4/02_drug_disease_associations/01_lincs_projection_archs4') / 'lincs' / 'lincs-projection.pkl'
display(LINCS_PROJ_FILE)
assert LINCS_PROJ_FILE.exists()

SPREDIXCAN_PROJ_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/00_spredixcan_projection_archs4') / 'spredixcan' / 'proj'
display(SPREDIXCAN_PROJ_DIR)
assert SPREDIXCAN_PROJ_DIR.exists()

OUTPUT_PREDICTIONS_DIR = OUTPUT_DIR / 'lincs' / 'predictions'
OUTPUT_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_PREDICTIONS_DIR)


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/drug_disease_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/01_lincs_projection_archs4/lincs/lincs-projection.pkl')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/00_spredixcan_projection_archs4/spredixcan/proj')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions')

# Helper functions


In [5]:
import sys
sys.path.insert(0, str(here('libs')))
from drug_disease_utils import map_traits_to_doid, _zero_nontop_genes, predict_dotprod_neg

# Load PharmacotherapyDB gold standard

In [6]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

doids_in_gold_standard = set(gold_standard['trait'])
print(f'Unique DOIDs in gold standard: {len(doids_in_gold_standard)}')

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


Unique DOIDs in gold standard: 87


# Load trait → DOID mapping files

In [ ]:
ukb_efo = pd.read_csv(
    DATA_DIR / 'phenomexcan_traits_fullcode_to_efo.tsv',
    sep='\t',
    index_col='ukb_fullcode',
)
# PhenoPlier stores trait full codes with hyphens (e.g. "I70-Diagnoses_...") but
# our S-PrediXcan data uses underscores throughout (e.g. "I70_Diagnoses_...").
# Normalize the index
ukb_efo.index = [idx.replace('-', '_', 1) for idx in ukb_efo.index]

efo_xrefs = pd.read_csv(DATA_DIR / 'term_id_xrefs.tsv.gz', sep='\t')
do_xrefs = pd.read_csv(DATA_DIR / 'xrefs-prop-slim.tsv', sep='\t')

# Load LINCS projection

In [8]:
input_file = LINCS_PROJ_FILE
display(input_file)
lincs_projection = pd.read_pickle(input_file)
display(lincs_projection.shape)
display(lincs_projection.head())

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/01_lincs_projection_archs4/lincs/lincs-projection.pkl')

(1728, 1170)

perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,0.003127,-0.014655,0.006365,-0.054058,0.059773,-0.023714,-0.049853,-0.023722,-0.001287,-0.015632,...,-0.079593,-0.059146,0.000345,0.034516,-0.000666,0.005783,0.029754,-0.172810,0.020110,0.000471
LV2,0.018831,0.045189,0.000370,-0.011590,0.000117,0.001920,-0.003038,-0.008572,-0.006517,0.002992,...,0.011073,-0.004997,0.017900,0.001621,0.014665,-0.001264,-0.008870,-0.049142,-0.007135,0.004529
LV3,-0.005911,-0.028553,0.005108,-0.000609,0.001949,0.004189,0.008837,0.005817,0.004727,0.002902,...,-0.003031,-0.000407,-0.007506,-0.013602,-0.006100,-0.009235,-0.000900,0.004442,0.005198,-0.002058
LV4,0.022497,-0.185666,0.002055,0.000259,0.027752,0.042420,-0.016758,0.029296,-0.013967,-0.021628,...,0.010162,-0.049056,0.036576,0.012428,-0.010925,-0.005625,-0.027448,-0.277678,0.008541,-0.012307
LV5,-0.001825,0.005136,0.002183,-0.006718,0.012377,-0.007434,0.019347,0.007425,-0.003604,0.002773,...,-0.001932,-0.003488,0.000656,0.008316,-0.002905,0.007354,-0.004349,-0.034426,0.000239,-0.000238


# Load S-PrediXcan projected files

In [9]:
spredixcan_file_list = sorted(
    f for f in SPREDIXCAN_PROJ_DIR.glob('spredixcan-*-projection-archs4.pkl')
)
display(len(spredixcan_file_list))
assert len(spredixcan_file_list) == 49

49

In [10]:
display(pd.read_pickle(spredixcan_file_list[0]).head())

,100001_raw_Food_weight,100002_raw_Energy,100003_raw_Protein,100004_raw_Fat,100005_raw_Carbohydrate,100006_raw_Saturated_fat,100007_raw_Polyunsaturated_fat,100008_raw_Total_sugars,100009_raw_Englyst_dietary_fibre,100010_Portion_size,...,Z50_Diagnoses_main_ICD10_Z50_Care_involving_use_of_rehabilitation_procedures,Z51_Diagnoses_main_ICD10_Z51_Other_medical_care,Z52_Diagnoses_main_ICD10_Z52_Donors_of_organs_and_tissues,Z53_Diagnoses_main_ICD10_Z53_Persons_encountering_health_services_for_specifie_procedures_not_carried_out,Z71_Diagnoses_main_ICD10_Z71_Persons_encountering_health_services_for_other_counselling_and_medical_advice_not_elsewhere_classified,Z76_Diagnoses_main_ICD10_Z76_Persons_encountering_health_services_in_other_circumstances,Z80_Diagnoses_main_ICD10_Z80_Family_history_of_malignant_neoplasm,Z85_Diagnoses_main_ICD10_Z85_Personal_history_of_malignant_neoplasm,Z87_Diagnoses_main_ICD10_Z87_Personal_history_of_other_diseases_and_conditions,pgc_scz2
LV1,0.006632,0.019139,0.018098,0.026516,0.006892,0.025451,0.011870,0.007200,0.001159,-0.003034,...,-0.007911,0.011209,-0.001099,0.008284,-0.006602,0.009399,-0.026363,-0.006694,0.010187,-0.003889
LV2,0.000411,-0.005822,0.000221,-0.006216,-0.004451,-0.011044,-0.006216,-0.004972,0.006376,0.008376,...,-0.023543,0.026545,0.030599,0.008681,-0.018226,-0.006836,-0.014249,0.003266,0.013255,0.019325
LV3,-0.009954,-0.022962,-0.023338,-0.014600,-0.023058,-0.006182,-0.028216,-0.004355,-0.020700,-0.027776,...,0.003069,0.020004,-0.011705,0.019495,0.004459,0.011248,-0.005372,-0.007361,-0.023956,-0.016283
LV4,-0.017000,-0.025840,-0.015511,-0.005718,-0.031565,-0.010693,0.004389,-0.027480,-0.015478,0.010452,...,0.016123,-0.012912,-0.037296,-0.017502,-0.028280,0.034280,-0.004154,0.024050,-0.006204,-0.008514
LV5,0.002084,0.004516,-0.000043,0.003571,0.001886,0.004024,0.001433,0.000660,-0.002262,0.004159,...,0.007271,0.002732,-0.000424,-0.007197,-0.010672,-0.002000,0.004713,0.002302,0.001561,-0.007655


# Predict drug-disease associations

In [ ]:
N_TOP_LVS_LIST = [None, 5, 10, 25, 50]

for spredixcan_file in spredixcan_file_list:
    print(spredixcan_file.name)

    # Load CLAMP-projected tissue-specific S-PrediXcan
    tissue_proj = pd.read_pickle(spredixcan_file)
    print(f'  shape: {tissue_proj.shape}')

    for ntc in N_TOP_LVS_LIST:
        predict_dotprod_neg(
            lincs_projection,
            spredixcan_file,
            tissue_proj,
            OUTPUT_PREDICTIONS_DIR,
            PREDICTION_METHOD,
            doids_in_gold_standard,
            ukb_efo,
            efo_xrefs,
            do_xrefs,
            n_top_conditions=ntc,
            use_abs=True,
        )

    print('')

spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-archs4.pkl


  shape: (1728, 4091)
  predicting all_genes...
    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-archs4-all_genes-prediction_scores.h5
  predicting top_5_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-archs4-top_5_genes-prediction_scores.h5
  predicting top_10_genes...


...

    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-archs4-top_10_genes-prediction_scores.h5
  predicting top_25_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-archs4-top_25_genes-prediction_scores.h5
  predicting top_50_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-archs4-top_50_genes-prediction_scores.h5

